In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
from src.data_collection.lob_reconstructor import *
from src.features.ofi_features import compute_all_features

plt.style.use('dark_background')
%matplotlib inline

RUN_ID   = "YOUR_RUN_ID_HERE"
DATA_DIR = Path("../data")

depth  = load_depth_data(RUN_ID, DATA_DIR)
trades = load_trade_data(RUN_ID, DATA_DIR)
depth  = add_derived_columns(depth)
merged = merge_depth_and_trades(depth, trades)
df     = compute_all_features(merged)

OFI Distribution:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['ofi_combined'].dropna(), bins=100, color='#00bfff', alpha=0.8)
axes[0].set_title('OFI Distribution')
axes[0].set_xlabel('OFI (combined)')

axes[1].plot(df['datetime'][:5000], df['ofi_zscore'][:5000], lw=0.5, color='#00bfff', alpha=0.7)
axes[1].axhline(0,    color='white', lw=0.5)
axes[1].axhline(1.5,  color='#ff4444', lw=0.8, ls='--', label='Entry threshold')
axes[1].axhline(-1.5, color='#ff4444', lw=0.8, ls='--')
axes[1].set_title('OFI Z-Score (first 5000 snapshots)')
axes[1].legend()
plt.tight_layout()
plt.show()

The key chart — OFI decile vs forward return:

In [ ]:
clean = df[['ofi_combined', 'target_return']].dropna()
clean['decile'] = pd.qcut(clean['ofi_combined'], 10, labels=False, duplicates='drop')
decile_means = clean.groupby('decile')['target_return'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Decile bar chart
colors = ['#ff4444' if x < 0 else '#00ff88' for x in decile_means]
axes[0].bar(range(len(decile_means)), decile_means.values, color=colors, alpha=0.85)
axes[0].axhline(0, color='white', lw=0.5)
axes[0].set_xlabel('OFI Decile (0=most negative, 9=most positive)')
axes[0].set_ylabel('Mean forward return')
axes[0].set_title('Mean Return by OFI Decile\n← Should be a staircase →')

# Scatter with regression line
sample = clean.sample(min(3000, len(clean)), random_state=42)
slope, intercept, r, p, _ = stats.linregress(clean['ofi_combined'], clean['target_return'])
x = np.linspace(clean['ofi_combined'].quantile(0.01), clean['ofi_combined'].quantile(0.99), 100)

axes[1].scatter(sample['ofi_combined'], sample['target_return'], alpha=0.15, s=4, color='#00bfff')
axes[1].plot(x, intercept + slope*x, 'r-', lw=2, label=f'β={slope:.5f}\nR²={r**2:.4f}\np={p:.2e}')
axes[1].set_xlabel('OFI combined')
axes[1].set_ylabel('Forward return')
axes[1].set_title('OFI vs Forward Return')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/02_ofi_vs_returns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOFI-Return correlation : {clean['ofi_combined'].corr(clean['target_return']):.4f}")
print(f"OLS β (Kyle's lambda)  : {slope:.6f}")
print(f"p-value                : {p:.4e}")